# BirdCLEF 2026 - 模型推理

本 Notebook 完成以下任务：
1. 定义模型架构（与训练时完全一致）
2. 定义测试数据加载逻辑
3. 加载训练好的模型权重
4. 在测试集上运行推理
5. 生成 `submission.csv`

In [ ]:
# ============================================================
# 0. 配置参数（必须与训练时一致）
# ============================================================

CHECKPOINT_PATH = "checkpoints/best.pt"   # 训练好的模型权重路径
TEST_AUDIO_DIR  = "test_soundscapes/"     # 测试音频目录（目前为空，赛中更新）
SUBMISSION_CSV  = "sample_submission.csv"  # 提交格式参考
OUTPUT_PATH     = "submission.csv"          # 推理结果输出路径

# 音频参数（与 src/utils.py 一致）
SAMPLE_RATE     = 32000
WINDOW_DURATION = 5.0       # 秒
N_MELS          = 128
N_FFT           = 2048
HOP_LENGTH      = 512
FMAX            = 16000
N_CLASSES       = 234       # 物种类别数
BACKBONE        = "resnet34" # 必须与训练时一致
DROPOUT         = 0.3       # 必须与训练时一致
BATCH_SIZE      = 64
DEVICE          = "cuda" if __import__("torch").cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
# ============================================================
# 1. 模型定义（内联，不依赖项目内部模块）
# ============================================================
import torch
import torch.nn as nn
from torchvision import models


class SpectrogramClassifier(nn.Module):
    """音频分类器：预训练 ResNet/EfficientNet 骨干 + 分类头。"""

    def __init__(self, n_classes: int, backbone: str = "resnet34",
                 pretrained: bool = True, dropout: float = 0.3):
        super().__init__()
        self.n_classes = n_classes

        if backbone == "resnet18":
            self.backbone = models.resnet18(
                weights="IMAGENET1K_V1" if pretrained else None)
            in_features = self.backbone.fc.in_features
        elif backbone == "resnet34":
            self.backbone = models.resnet34(
                weights="IMAGENET1K_V1" if pretrained else None)
            in_features = self.backbone.fc.in_features
        elif backbone == "resnet50":
            self.backbone = models.resnet50(
                weights="IMAGENET1K_V1" if pretrained else None)
            in_features = self.backbone.fc.in_features
        elif backbone == "efficientnet_b0":
            self.backbone = models.efficientnet_b0(
                weights="IMAGENET1K_V1" if pretrained else None)
            in_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()
        else:
            raise ValueError(f"Unknown backbone: {backbone}")

        self.backbone_name = backbone

        if "efficientnet" not in backbone:
            # ResNet: 修改第一层卷积接受单通道输入
            old_conv = self.backbone.conv1
            self.backbone.conv1 = nn.Conv2d(
                1, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False)
            with torch.no_grad():
                self.backbone.conv1.weight = nn.Parameter(
                    old_conv.weight.mean(dim=1, keepdim=True))
            self.backbone.fc = nn.Identity()

            self.fc = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(in_features, 512),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(512, n_classes),
            )
        else:
            # EfficientNet: 复制单通道为 3 通道
            self.fc = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(in_features, 512),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(512, n_classes),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, 1, n_mels, time) mel 频谱图
        Returns:
            (batch, n_classes) logits
        """
        if self.backbone_name == "efficientnet_b0":
            x = x.repeat(1, 3, 1, 1)
            features = self.backbone(x)
            return self.fc(features)
        else:
            features = self.backbone(x)
            return self.fc(features)


class MultiLabelClassifier(nn.Module):
    """多标签分类包装器，输出原始 logits。"""

    def __init__(self, n_classes: int, backbone: str = "resnet34",
                 pretrained: bool = True, dropout: float = 0.3):
        super().__init__()
        self.classifier = SpectrogramClassifier(
            n_classes=n_classes, backbone=backbone,
            pretrained=pretrained, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)


# 实例化模型
model = MultiLabelClassifier(
    n_classes=N_CLASSES,
    backbone=BACKBONE,
    pretrained=False,
    dropout=DROPOUT
).to(DEVICE)

# 统计参数量
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {BACKBONE}, parameters: {n_params:,}")

In [ ]:
# ============================================================
# 2. 加载模型权重
# ============================================================
import os

if os.path.exists(CHECKPOINT_PATH):
    state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    print(f"✓ 已加载权重: {CHECKPOINT_PATH}")
else:
    print(f"⚠ 权重文件不存在: {CHECKPOINT_PATH}，将使用随机权重推理（仅用于测试流程）")

model.eval()
print("模型已切换到评估模式")

In [ ]:
# ============================================================
# 3. 测试数据加载（InferenceDataset）
# ============================================================
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from torch.utils.data import Dataset


def pad_or_truncate(audio: np.ndarray, target_samples: int) -> np.ndarray:
    """将音频填充或截断到目标长度。"""
    if len(audio) < target_samples:
        audio = np.pad(audio, (0, target_samples - len(audio)))
    elif len(audio) > target_samples:
        audio = audio[:target_samples]
    return audio


def load_audio(path, sr=32000, offset=0.0, duration=None):
    """加载音频文件。"""
    audio, _ = librosa.load(path, sr=sr, offset=offset, duration=duration)
    return audio


def audio_to_melspec(audio, sr=32000, n_mels=128, n_fft=2048,
                     hop_length=512):
    """将音频转为 Mel 频谱图（dB 尺度）。"""
    spec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length,
        n_mels=n_mels, fmin=0, fmax=16000, power=2.0)
    return librosa.power_to_db(spec, ref=np.max)


def process_fixed_window(path, sr, n_mels, n_fft, hop_length,
                        window_duration, offset=0.0):
    """加载音频 → 填充/截断 → 转 Mel 频谱图。"""
    audio = load_audio(path, sr=sr, offset=offset, duration=window_duration)
    target_samples = int(sr * window_duration)
    audio = pad_or_truncate(audio, target_samples)
    spec = audio_to_melspec(audio, sr=sr, n_mels=n_mels,
                             n_fft=n_fft, hop_length=hop_length)
    return spec


class InferenceDataset(Dataset):
    """测试数据集：根据 row_id 加载对应音频窗口。"""

    def __init__(self, soundscapes_dir, row_ids,
                 sr=32000, n_mels=128, n_fft=2048,
                 hop_length=512, window_duration=5.0):
        self.soundscapes_dir = Path(soundscapes_dir)
        self.row_ids = row_ids
        self.sr = sr
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.window_duration = window_duration
        # 计算时间帧数（用于生成零频谱图）
        self.n_time = int(sr * window_duration / hop_length) + 1

    @staticmethod
    def parse_row_id(row_id: str):
        """解析 row_id → (filename, offset_seconds)。"""
        # row_id 格式: BC2026_Test_XXXX_{offset}  或  BC2026_Train_XXXX_{offset}
        parts = row_id.rsplit("_", 1)
        offset = float(parts[1])
        filename = parts[0] + ".ogg"
        return filename, offset

    def __len__(self):
        return len(self.row_ids)

    def __getitem__(self, idx):
        row_id = self.row_ids[idx]
        filename, offset = self.parse_row_id(row_id)
        path = self.soundscapes_dir / filename

        if path.exists():
            spec = process_fixed_window(
                path,
                sr=self.sr, n_mels=self.n_mels,
                n_fft=self.n_fft, hop_length=self.hop_length,
                window_duration=self.window_duration,
                offset=offset)
            spec = (spec - spec.mean()) / (spec.std() + 1e-6)
        else:
            # 文件不存在时返回零频谱图（比赛中不应出现）
            spec = np.zeros((self.n_mels, self.n_time))

        spec_tensor = torch.from_numpy(spec).float().unsqueeze(0)  # (1, n_mels, time)
        return spec_tensor, row_id


# 读取提交格式
sub_df = pd.read_csv(SUBMISSION_CSV)
species_list = sub_df.columns[1:].tolist()  # row_id 之后的 234 列
row_ids = sub_df["row_id"].tolist()
print(f"提交格式: {len(row_ids)} 个窗口, {len(species_list)} 个物种")
print(f"前 3 个 row_id: {row_ids[:3]}")

# 检查测试音频目录是否有效
test_dir = Path(TEST_AUDIO_DIR)
test_files = list(test_dir.glob("*.ogg"))
print(f"\n测试音频目录: {test_dir}")
print(f"音频文件数量: {len(test_files)}")
if not test_files:
    print("⚠ 测试目录为空，比赛中测试数据将在 rerun 时更新。")
    print("   如需用训练集验证推理流程，可将 TEST_AUDIO_DIR 改为 'train_soundscapes/'。")

In [ ]:
# ============================================================
# 4. 运行推理
# ============================================================
from torch.utils.data import DataLoader
from tqdm import tqdm

# 创建数据集和 DataLoader
dataset = InferenceDataset(
    soundscapes_dir=TEST_AUDIO_DIR,
    row_ids=row_ids,
    sr=SAMPLE_RATE,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    window_duration=WINDOW_DURATION
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=(DEVICE == "cuda")
)

print(f"数据集大小: {len(dataset)}")
print(f"Batch size: {BATCH_SIZE}, 总批次数: {len(loader)}")

# 推理循环
all_row_ids = []
all_probs = []

with torch.no_grad():
    for batch_idx, (specs, batch_row_ids) in enumerate(tqdm(loader, desc="推理中")):
        specs = specs.to(DEVICE)
        logits = model(specs)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_row_ids.extend(batch_row_ids)
        all_probs.append(probs)

        if (batch_idx + 1) % 20 == 0:
            print(f"  Batch {batch_idx + 1}/{len(loader)}")

all_probs = np.vstack(all_probs)
print(f"\n推理完成，共 {len(all_row_ids)} 个窗口，概率矩阵 shape: {all_probs.shape}")

In [ ]:
# ============================================================
# 5. 生成 submission.csv
# ============================================================

# 构建 DataFrame: row_id + 234 个物种概率列
submission = pd.DataFrame(all_probs, columns=species_list)
submission.insert(0, "row_id", all_row_ids)

# 保存
submission.to_csv(OUTPUT_PATH, index=False)
print(f"✓ 已保存到: {OUTPUT_PATH}")
print(f"  Shape: {submission.shape}")
print()
print("前 5 行预览:")
display(submission.head())

In [ ]:
# ============================================================
# 6. 质量检查
# ============================================================

# 检查概率分布
print("=== 概率值统计 ===")
prob_values = all_probs.ravel()
print(f"  最小值: {prob_values.min():.6f}")
print(f"  最大值: {prob_values.max():.6f}")
print(f"  均值:   {prob_values.mean():.6f}")
print(f"  标准差: {prob_values.std():.6f}")

# 检查每行有多少物种被预测为正（概率 > 0.5）
positive_per_row = (all_probs > 0.5).sum(axis=1)
print(f"\n=== 每窗口正类数量 ===")
print(f"  最小: {positive_per_row.min()}, 最大: {positive_per_row.max()}, 均值: {positive_per_row.mean():.1f}")

# 检查每个物种的正样本比例
positive_per_species = (all_probs > 0.5).sum(axis=0)
top_species = np.argsort(positive_per_species)[::-1][:10]
print(f"\n=== 最常被预测的 Top 10 物种 ===")
for idx in top_species:
    print(f"  {species_list[idx]}: {positive_per_species[idx]} 窗口为正 ({100*positive_per_species[idx]/len(all_probs):.1f}%)")